
# Segmentação de Clientes por Segmento Interno

Este notebook adapta uma clusterização genérica para uma segmentação bancária acionável, classificando cada cliente em um dos quatro arquétipos:

1. **Top Performance** — alto valor atual, boa penetração e risco controlado.
2. **Foco em Conversão** — alto potencial econômico, mas baixa captura de valor ou baixa penetração de produtos.
3. **Crescimento em Valor** — trajetória positiva de ROB, produtos ou engajamento.
4. **Em Risco** — pior rating/restrição, queda de resultado ou sinais de deterioração.

## Princípio essencial

Os clientes são comparados **somente com outros clientes do mesmo segmento interno**. Assim, um MEI não é avaliado com a mesma régua de uma Empresa de maior porte.

O modelo usa quatro escores interpretáveis — valor atual, potencial, crescimento e risco — e aplica uma clusterização separada por segmento. Depois, os clusters numéricos são associados a protótipos de negócio para manter nomes consistentes.


In [ ]:

# Caso necessário, instale as dependências:
# %pip install pandas numpy scikit-learn scipy matplotlib pyarrow openpyxl

from pathlib import Path
import warnings
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from threadpoolctl import threadpool_limits

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)



## 1. Configuração

Altere apenas o caminho da base e o dicionário `COLUNAS`.

As colunas de mês anterior são opcionais. Quando não existirem, o componente de crescimento ficará neutro até que uma janela histórica seja adicionada.


In [ ]:

ARQUIVO_BASE = Path("base_clientes.csv")
USAR_DADOS_EXEMPLO = False

# Nome lógico utilizado pelo modelo: nome da coluna na sua base
COLUNAS = {
    "id_cliente": "CNPJ",
    "segmento": "MAIOR_SUBSEGMENTO_AJUSTADO",
    "rob_atual": "ROB_ATUAL",
    "rob_anterior": "ROB_ANTERIOR",                 # opcional
    "faturamento": "FATURAMENTO",
    "qtd_produtos": "QTD_PRODUTOS",                 # pode ser calculada pelas flags POSSUI_
    "qtd_produtos_anterior": "QTD_PRODUTOS_ANTERIOR", # opcional
    "engajamento": "NIVEL_ENGAJAMENTO",             # opcional
    "rating": "RATING",                             # recomendado
    "grau_restricao": "GRAU_RESTRICAO",             # recomendado
}

PREFIXO_FLAGS_PRODUTOS = "POSSUI_"
NORMALIZAR_ID_COMO_CNPJ = True

# Segmentos muito pequenos são classificados diretamente pelos protótipos,
# evitando um K-Means instável.
MIN_CLIENTES_PARA_CLUSTER = 40

# Amostra máxima usada para treinar o MiniBatchKMeans dentro de cada segmento.
# Todos os clientes recebem a predição após o ajuste.
MAX_AMOSTRA_TREINO_SEGMENTO = 100_000
MAX_AMOSTRA_SILHUETA = 3_000

SALVAR_CSV_COMPLETO = False


## 2. Funções auxiliares

In [ ]:

def gerar_dados_exemplo(n=12_000, random_state=42):
    rng = np.random.default_rng(random_state)
    segmentos = rng.choice(
        ["NEGÓCIOS MEI", "NEGÓCIOS PJ", "EMPRESAS"],
        size=n,
        p=[0.45, 0.40, 0.15]
    )

    fator_segmento = pd.Series(segmentos).map({
        "NEGÓCIOS MEI": 0.45,
        "NEGÓCIOS PJ": 1.00,
        "EMPRESAS": 2.30
    }).to_numpy()

    faturamento = rng.lognormal(mean=11.0, sigma=1.0, size=n) * fator_segmento
    qtd_produtos = np.clip(
        rng.poisson(lam=3.2 * np.sqrt(fator_segmento), size=n), 0, 20
    )
    rob_anterior = (
        faturamento * rng.uniform(0.0008, 0.008, size=n)
        + qtd_produtos * rng.uniform(50, 250, size=n)
    )
    tendencia = rng.normal(0.04, 0.25, size=n)
    rob_atual = rob_anterior * (1 + tendencia)
    qtd_produtos_anterior = np.clip(
        qtd_produtos - rng.choice([0, 0, 0, 1, -1], size=n),
        0,
        None
    )

    rating = rng.choice(
        ["AA", "A", "B", "C", "D", "E", "F", "G", "H"],
        size=n,
        p=[0.12, 0.20, 0.23, 0.18, 0.11, 0.07, 0.04, 0.03, 0.02]
    )
    grau_restricao = np.clip(
        np.round(
            pd.Series(rating).map({
                "AA": 0, "A": 0, "B": 1, "C": 1,
                "D": 2, "E": 3, "F": 4, "G": 5, "H": 5
            }).to_numpy()
            + rng.normal(0, 0.8, n)
        ),
        0,
        5
    )
    engajamento = rng.choice(["0", "1", "2", "VIP"], size=n, p=[0.20, 0.35, 0.30, 0.15])

    return pd.DataFrame({
        "CNPJ": [str(10_000_000_000_000 + i) for i in range(n)],
        "MAIOR_SUBSEGMENTO_AJUSTADO": segmentos,
        "ROB_ATUAL": rob_atual,
        "ROB_ANTERIOR": rob_anterior,
        "FATURAMENTO": faturamento,
        "QTD_PRODUTOS": qtd_produtos,
        "QTD_PRODUTOS_ANTERIOR": qtd_produtos_anterior,
        "NIVEL_ENGAJAMENTO": engajamento,
        "RATING": rating,
        "GRAU_RESTRICAO": grau_restricao,
    })


def carregar_base(caminho, usar_exemplo=False):
    if usar_exemplo:
        return gerar_dados_exemplo()

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho.resolve()}\n"
            "Ajuste ARQUIVO_BASE ou defina USAR_DADOS_EXEMPLO = True."
        )

    extensao = caminho.suffix.lower()

    if extensao == ".csv":
        return pd.read_csv(caminho, sep=None, engine="python", dtype=str)
    if extensao in {".parquet", ".pq"}:
        return pd.read_parquet(caminho)
    if extensao in {".xlsx", ".xls"}:
        return pd.read_excel(caminho)

    raise ValueError("Formato não suportado. Use CSV, Parquet ou Excel.")


def normalizar_cnpj(serie):
    return (
        serie.astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(14)
    )


def rating_para_risco(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).upper().strip()
    texto = re.sub(r"[^A-Z]", "", texto)

    if texto.startswith("AA"):
        return 0.0

    mapa = {letra: indice for indice, letra in enumerate("ABCDEFGH", start=1)}
    for letra, valor_risco in mapa.items():
        if texto.startswith(letra):
            return float(valor_risco)

    return np.nan


def engajamento_para_numero(valor):
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)

    texto = str(valor).upper().strip().replace("_", " ")

    if texto.startswith("VIP"):
        numero = re.findall(r"\d+", texto)
        return 3.0 + (float(numero[0]) / 10 if numero else 0.0)

    numero = re.findall(r"-?\d+(?:[.,]\d+)?", texto)
    if numero:
        return float(numero[0].replace(",", "."))

    return np.nan


def numerica(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")

    texto = serie.astype("string").str.strip()

    # Trata formato brasileiro quando houver vírgula decimal.
    contem_virgula = texto.str.contains(",", regex=False, na=False).mean() > 0.20
    if contem_virgula:
        texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)

    return pd.to_numeric(texto, errors="coerce")


def percentil_no_segmento(serie, segmento):
    x = numerica(serie).replace([np.inf, -np.inf], np.nan)

    mediana_segmento = x.groupby(segmento).transform("median")
    x = x.fillna(mediana_segmento)
    x = x.fillna(x.median()).fillna(0.0)

    percentil = x.groupby(segmento).rank(method="average", pct=True)
    quantidade_valores = x.groupby(segmento).transform("nunique")

    return percentil.where(quantidade_valores > 1, 0.5).clip(0, 1)


def media_ponderada(componentes):
    numerador = None
    denominador = 0.0

    for serie, peso in componentes:
        parcela = serie.astype(float) * peso
        numerador = parcela if numerador is None else numerador + parcela
        denominador += peso

    return (numerador / denominador).clip(0, 1)


## 3. Carregamento e preparação da base

In [ ]:

df_original = carregar_base(ARQUIVO_BASE, USAR_DADOS_EXEMPLO)
df = df_original.copy()

# Cria cópias com nomes lógicos, preservando as colunas originais.
for nome_logico, nome_origem in COLUNAS.items():
    if nome_origem in df.columns:
        df[nome_logico] = df[nome_origem]

# Calcula quantidade de produtos pelas flags, caso a coluna pronta não exista.
flags_produtos = [
    coluna for coluna in df.columns
    if coluna.upper().startswith(PREFIXO_FLAGS_PRODUTOS.upper())
]

if "qtd_produtos" not in df.columns and flags_produtos:
    df["qtd_produtos"] = (
        df[flags_produtos]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
    )

essenciais = ["id_cliente", "segmento", "rob_atual", "faturamento", "qtd_produtos"]
faltantes = [coluna for coluna in essenciais if coluna not in df.columns]

if faltantes:
    raise ValueError(
        "Colunas essenciais não encontradas: "
        + ", ".join(faltantes)
        + ". Revise o dicionário COLUNAS."
    )

# Campos opcionais recebem valores neutros ou equivalentes.
if "rob_anterior" not in df.columns:
    df["rob_anterior"] = df["rob_atual"]

if "qtd_produtos_anterior" not in df.columns:
    df["qtd_produtos_anterior"] = df["qtd_produtos"]

if "engajamento" not in df.columns:
    df["engajamento"] = np.nan

if "rating" not in df.columns:
    df["rating"] = np.nan

if "grau_restricao" not in df.columns:
    df["grau_restricao"] = np.nan

df["segmento"] = (
    df["segmento"]
    .astype("string")
    .fillna("SEM_SEGMENTO")
    .str.strip()
    .replace("", "SEM_SEGMENTO")
)

if NORMALIZAR_ID_COMO_CNPJ:
    df["id_cliente"] = normalizar_cnpj(df["id_cliente"])
else:
    df["id_cliente"] = df["id_cliente"].astype("string")

colunas_numericas = [
    "rob_atual", "rob_anterior", "faturamento",
    "qtd_produtos", "qtd_produtos_anterior", "grau_restricao"
]

for coluna in colunas_numericas:
    df[coluna] = numerica(df[coluna])

df["engajamento_num"] = df["engajamento"].map(engajamento_para_numero)
df["rating_risco_num"] = df["rating"].map(rating_para_risco)

print(f"Linhas: {len(df):,}")
print(f"Segmentos: {df['segmento'].nunique():,}")
display(df[[
    "id_cliente", "segmento", "rob_atual", "faturamento",
    "qtd_produtos", "rating", "grau_restricao"
]].head())



## 4. Tratamento de extremos e engenharia de atributos

Em vez de excluir clientes extremos — que podem ser justamente os clientes mais relevantes — os valores são limitados nos percentis 1% e 99% **dentro de cada segmento**.

Depois, cada indicador é convertido em percentil dentro do próprio segmento. Isso cria uma régua de 0 a 1 e reduz a influência da escala absoluta.


In [ ]:

def winsorizar_por_segmento(dataframe, coluna, inferior=0.01, superior=0.99):
    valores = dataframe[coluna].copy()

    limite_inferior = valores.groupby(dataframe["segmento"]).transform(
        lambda s: s.quantile(inferior)
    )
    limite_superior = valores.groupby(dataframe["segmento"]).transform(
        lambda s: s.quantile(superior)
    )

    return valores.clip(lower=limite_inferior, upper=limite_superior)


for coluna in [
    "rob_atual", "rob_anterior", "faturamento",
    "qtd_produtos", "qtd_produtos_anterior",
    "engajamento_num", "rating_risco_num", "grau_restricao"
]:
    df[f"{coluna}_tratado"] = winsorizar_por_segmento(df, coluna)

df["variacao_rob"] = (
    (df["rob_atual_tratado"] - df["rob_anterior_tratado"])
    / (df["rob_anterior_tratado"].abs() + 1.0)
).clip(-2, 2)

df["variacao_produtos"] = (
    df["qtd_produtos_tratado"] - df["qtd_produtos_anterior_tratado"]
).clip(-10, 10)

# Percentis comparáveis somente dentro de cada segmento.
df["p_rob"] = percentil_no_segmento(df["rob_atual_tratado"], df["segmento"])
df["p_faturamento"] = percentil_no_segmento(df["faturamento_tratado"], df["segmento"])
df["p_produtos"] = percentil_no_segmento(df["qtd_produtos_tratado"], df["segmento"])
df["p_engajamento"] = percentil_no_segmento(df["engajamento_num_tratado"], df["segmento"])
df["p_crescimento_rob"] = percentil_no_segmento(df["variacao_rob"], df["segmento"])
df["p_crescimento_produtos"] = percentil_no_segmento(df["variacao_produtos"], df["segmento"])
df["p_rating_risco"] = percentil_no_segmento(df["rating_risco_num_tratado"], df["segmento"])
df["p_restricao"] = percentil_no_segmento(df["grau_restricao_tratado"], df["segmento"])



## 5. Construção dos quatro escores

Os pesos abaixo são uma primeira versão de negócio e devem ser validados com especialistas.

- **Valor atual:** ROB, quantidade de produtos e engajamento.
- **Potencial:** faturamento alto combinado com baixa captura de ROB/produtos e risco controlado.
- **Crescimento:** evolução de ROB, evolução de produtos e engajamento.
- **Risco:** rating, restrição e deterioração do ROB.


In [ ]:

# Ajuste estes pesos após validar os centroides e os resultados com a área de negócio.
df["score_valor_atual"] = media_ponderada([
    (df["p_rob"], 0.50),
    (df["p_produtos"], 0.30),
    (df["p_engajamento"], 0.20),
])

# Primeiro calculamos o risco para usá-lo também no potencial.
df["score_risco"] = media_ponderada([
    (df["p_rating_risco"], 0.45),
    (df["p_restricao"], 0.35),
    (1 - df["p_crescimento_rob"], 0.20),
])

df["score_potencial"] = media_ponderada([
    (df["p_faturamento"], 0.45),
    (1 - df["p_produtos"], 0.25),
    (1 - df["p_rob"], 0.20),
    (1 - df["score_risco"], 0.10),
])

df["score_crescimento"] = media_ponderada([
    (df["p_crescimento_rob"], 0.60),
    (df["p_crescimento_produtos"], 0.25),
    (df["p_engajamento"], 0.15),
])

SCORE_COLS = [
    "score_valor_atual",
    "score_potencial",
    "score_crescimento",
    "score_risco",
]

display(df.groupby("segmento")[SCORE_COLS].agg(["mean", "median", "std"]).round(3))



## 6. Protótipos de negócio

O K-Means gera números sem significado fixo. Para evitar que o Cluster 0 mude de nome entre segmentos, os centroides são associados aos protótipos abaixo pela menor distância global.

Esses protótipos não classificam diretamente os segmentos grandes; eles apenas dão um nome estável aos quatro clusters encontrados.


In [ ]:

PROTOTIPOS = pd.DataFrame(
    {
        "score_valor_atual": [0.85, 0.30, 0.55, 0.35],
        "score_potencial":   [0.50, 0.85, 0.60, 0.35],
        "score_crescimento": [0.65, 0.45, 0.85, 0.20],
        "score_risco":       [0.15, 0.25, 0.25, 0.85],
    },
    index=[
        "Top Performance",
        "Foco em Conversão",
        "Crescimento em Valor",
        "Em Risco",
    ],
)

display(PROTOTIPOS)


## 7. Clusterização separada por segmento

In [ ]:

def mapear_clusters_para_perfis(centroides):
    distancia = cdist(
        centroides[SCORE_COLS].to_numpy(),
        PROTOTIPOS[SCORE_COLS].to_numpy()
    )

    linhas, colunas = linear_sum_assignment(distancia)

    return {
        centroides.index[linha]: PROTOTIPOS.index[coluna]
        for linha, coluna in zip(linhas, colunas)
    }


def classificar_por_prototipo(dados_segmento):
    distancia = cdist(
        dados_segmento[SCORE_COLS].to_numpy(),
        PROTOTIPOS[SCORE_COLS].to_numpy()
    )
    indice_melhor = distancia.argmin(axis=1)

    return (
        indice_melhor,
        PROTOTIPOS.index.to_numpy()[indice_melhor]
    )


def clusterizar_segmento(dados_segmento):
    dados_segmento = dados_segmento.copy()
    nome_segmento = dados_segmento["segmento"].iloc[0]
    n_clientes = len(dados_segmento)
    n_combinacoes = dados_segmento[SCORE_COLS].drop_duplicates().shape[0]

    # Segmentos pequenos ou quase constantes: classificação por proximidade
    # aos protótipos, sem forçar quatro clusters instáveis.
    if n_clientes < MIN_CLIENTES_PARA_CLUSTER or n_combinacoes < 4:
        cluster_num, perfis = classificar_por_prototipo(dados_segmento)
        dados_segmento["cluster_numerico"] = cluster_num
        dados_segmento["perfil_cliente"] = perfis
        dados_segmento["metodo_classificacao"] = "prototipo_direto"

        diagnostico = {
            "segmento": nome_segmento,
            "clientes": n_clientes,
            "metodo": "prototipo_direto",
            "silhueta": np.nan,
        }
        return dados_segmento, diagnostico, None

    scaler = StandardScaler()
    X = scaler.fit_transform(dados_segmento[SCORE_COLS])

    if n_clientes > MAX_AMOSTRA_TREINO_SEGMENTO:
        rng = np.random.default_rng(RANDOM_STATE)
        indice_treino = rng.choice(
            n_clientes,
            size=MAX_AMOSTRA_TREINO_SEGMENTO,
            replace=False
        )
        X_treino = X[indice_treino]
    else:
        X_treino = X

    modelo = MiniBatchKMeans(
        n_clusters=4,
        random_state=RANDOM_STATE,
        n_init=20,
        batch_size=4096,
        reassignment_ratio=0.01
    )
    # Evita excesso de threads em notebooks locais e ambientes corporativos.
    with threadpool_limits(limits=1):
        modelo.fit(X_treino)
        clusters = modelo.predict(X)
    dados_segmento["cluster_numerico"] = clusters

    centroides = (
        dados_segmento
        .groupby("cluster_numerico")[SCORE_COLS]
        .mean()
    )
    mapa_perfis = mapear_clusters_para_perfis(centroides)

    dados_segmento["perfil_cliente"] = (
        dados_segmento["cluster_numerico"].map(mapa_perfis)
    )
    dados_segmento["metodo_classificacao"] = "minibatch_kmeans"

    # Diagnóstico em uma amostra para não sobrecarregar bases grandes.
    n_silhueta = min(n_clientes, MAX_AMOSTRA_SILHUETA)
    if n_silhueta < n_clientes:
        rng = np.random.default_rng(RANDOM_STATE)
        indice_diag = rng.choice(n_clientes, size=n_silhueta, replace=False)
        X_diag = X[indice_diag]
        y_diag = clusters[indice_diag]
    else:
        X_diag = X
        y_diag = clusters

    silhueta = (
        silhouette_score(X_diag, y_diag)
        if len(np.unique(y_diag)) > 1
        else np.nan
    )

    diagnostico = {
        "segmento": nome_segmento,
        "clientes": n_clientes,
        "metodo": "minibatch_kmeans",
        "silhueta": silhueta,
    }

    artefatos_modelo = {
        "scaler": scaler,
        "modelo": modelo,
        "mapa_perfis": mapa_perfis,
    }

    return dados_segmento, diagnostico, artefatos_modelo


In [ ]:

partes = []
diagnosticos = []
modelos_por_segmento = {}

for segmento, dados_segmento in df.groupby("segmento", sort=False):
    resultado_segmento, diagnostico, artefatos = clusterizar_segmento(dados_segmento)

    partes.append(resultado_segmento)
    diagnosticos.append(diagnostico)

    if artefatos is not None:
        modelos_por_segmento[segmento] = artefatos

df_segmentado = pd.concat(partes, ignore_index=True)
df_diagnostico = pd.DataFrame(diagnosticos).sort_values("clientes", ascending=False)

display(df_diagnostico)


## 8. Priorização e recomendação acionável

In [ ]:

acoes = {
    "Top Performance":
        "Proteger relacionamento, reconhecer valor e ampliar participação com ofertas premium.",
    "Foco em Conversão":
        "Ativar cross-sell e converter potencial econômico em produtos, fluxo e resultado.",
    "Crescimento em Valor":
        "Acelerar a expansão com limites, soluções de caixa, cobrança, TPV e capital de giro.",
    "Em Risco":
        "Atuar preventivamente, revisar deterioração, restrições, relacionamento e exposição.",
}

motivos = {
    "Top Performance":
        "Valor atual e penetração acima dos pares do segmento.",
    "Foco em Conversão":
        "Potencial elevado, mas captura de ROB ou produtos ainda baixa.",
    "Crescimento em Valor":
        "Evolução recente acima dos pares, com espaço para aprofundamento.",
    "Em Risco":
        "Risco ou deterioração de resultado acima dos pares do segmento.",
}

df_segmentado["acao_recomendada"] = df_segmentado["perfil_cliente"].map(acoes)
df_segmentado["motivo_principal"] = df_segmentado["perfil_cliente"].map(motivos)

# Prioridade específica por perfil.
condicoes = [
    df_segmentado["perfil_cliente"].eq("Top Performance"),
    df_segmentado["perfil_cliente"].eq("Foco em Conversão"),
    df_segmentado["perfil_cliente"].eq("Crescimento em Valor"),
    df_segmentado["perfil_cliente"].eq("Em Risco"),
]

formulas_prioridade = [
    0.50 * df_segmentado["score_valor_atual"]
    + 0.30 * df_segmentado["score_potencial"]
    + 0.20 * df_segmentado["score_crescimento"],

    0.55 * df_segmentado["score_potencial"]
    + 0.25 * (1 - df_segmentado["score_risco"])
    + 0.20 * df_segmentado["p_faturamento"],

    0.55 * df_segmentado["score_crescimento"]
    + 0.25 * df_segmentado["score_potencial"]
    + 0.20 * df_segmentado["score_valor_atual"],

    0.55 * df_segmentado["score_risco"]
    + 0.30 * df_segmentado["score_valor_atual"]
    + 0.15 * (1 - df_segmentado["score_crescimento"]),
]

df_segmentado["prioridade_raw"] = np.select(
    condicoes,
    formulas_prioridade,
    default=0.0
)

df_segmentado["percentil_prioridade"] = (
    df_segmentado
    .groupby(["segmento", "perfil_cliente"])["prioridade_raw"]
    .rank(method="average", pct=True)
)

df_segmentado["faixa_prioridade"] = np.select(
    [
        df_segmentado["percentil_prioridade"] >= 0.80,
        df_segmentado["percentil_prioridade"] >= 0.50,
    ],
    ["A - Alta", "B - Média"],
    default="C - Acompanhar"
)

df_segmentado["score_valor_atual_100"] = (100 * df_segmentado["score_valor_atual"]).round(1)
df_segmentado["score_potencial_100"] = (100 * df_segmentado["score_potencial"]).round(1)
df_segmentado["score_crescimento_100"] = (100 * df_segmentado["score_crescimento"]).round(1)
df_segmentado["score_risco_100"] = (100 * df_segmentado["score_risco"]).round(1)
df_segmentado["score_prioridade_100"] = (100 * df_segmentado["prioridade_raw"]).round(1)

display(df_segmentado[[
    "id_cliente", "segmento", "perfil_cliente", "faixa_prioridade",
    "score_valor_atual_100", "score_potencial_100",
    "score_crescimento_100", "score_risco_100",
    "motivo_principal", "acao_recomendada"
]].head(10))


## 9. Validação dos resultados

In [ ]:

resumo_perfis = (
    df_segmentado
    .groupby(["segmento", "perfil_cliente"], observed=True)
    .agg(
        clientes=("id_cliente", "nunique"),
        rob_total=("rob_atual", "sum"),
        rob_medio=("rob_atual", "mean"),
        faturamento_medio=("faturamento", "mean"),
        produtos_medio=("qtd_produtos", "mean"),
        score_valor=("score_valor_atual_100", "mean"),
        score_potencial=("score_potencial_100", "mean"),
        score_crescimento=("score_crescimento_100", "mean"),
        score_risco=("score_risco_100", "mean"),
    )
    .reset_index()
)

resumo_perfis["percentual_clientes"] = (
    resumo_perfis["clientes"]
    / resumo_perfis.groupby("segmento")["clientes"].transform("sum")
)

display(
    resumo_perfis.sort_values(
        ["segmento", "clientes"],
        ascending=[True, False]
    ).round(2)
)


In [ ]:

# Centroides finais por perfil: esta é a tabela principal para validação de negócio.
centroides_perfis = (
    df_segmentado
    .groupby(["segmento", "perfil_cliente"], observed=True)[SCORE_COLS]
    .mean()
    .reset_index()
)

display(centroides_perfis.round(3))



### O que deve ser verificado

1. **Top Performance** deve ter o maior `score_valor_atual` e risco controlado.
2. **Foco em Conversão** deve ter o maior `score_potencial`, normalmente com menor penetração.
3. **Crescimento em Valor** deve ter o maior `score_crescimento`.
4. **Em Risco** deve ter o maior `score_risco`.
5. A silhueta não precisa ser perfeita; o objetivo é criar grupos úteis e estáveis para ação comercial.
6. Se algum perfil não fizer sentido em determinado segmento, revise os pesos ou os protótipos antes de levar o resultado para produção.


## 10. Visualizações exploratórias

In [ ]:

SEGMENTO_PARA_PLOTAR = df_segmentado["segmento"].value_counts().index[0]
AMOSTRA_PLOT = 8_000

plot_df = df_segmentado[
    df_segmentado["segmento"].eq(SEGMENTO_PARA_PLOTAR)
].copy()

if len(plot_df) > AMOSTRA_PLOT:
    plot_df = plot_df.sample(AMOSTRA_PLOT, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(10, 7))

for perfil, grupo in plot_df.groupby("perfil_cliente"):
    ax.scatter(
        grupo["score_valor_atual_100"],
        grupo["score_potencial_100"],
        s=18,
        alpha=0.55,
        label=perfil
    )

ax.set_title(f"Valor atual x potencial — {SEGMENTO_PARA_PLOTAR}")
ax.set_xlabel("Score de valor atual")
ax.set_ylabel("Score de potencial")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:

perfil_segmento = (
    df_segmentado[
        df_segmentado["segmento"].eq(SEGMENTO_PARA_PLOTAR)
    ]
    .groupby("perfil_cliente")[SCORE_COLS]
    .mean()
    .T
)

ax = perfil_segmento.plot(figsize=(11, 6), marker="o")
ax.set_title(f"Perfil médio dos arquétipos — {SEGMENTO_PARA_PLOTAR}")
ax.set_ylabel("Score médio de 0 a 1")
ax.set_xlabel("Dimensão")
ax.set_ylim(0, 1)
ax.grid(alpha=0.2)
plt.xticks(rotation=0)
plt.show()


## 11. Exportação para Power BI

In [ ]:

colunas_modelo = [
    "id_cliente",
    "segmento",
    "perfil_cliente",
    "cluster_numerico",
    "metodo_classificacao",
    "faixa_prioridade",
    "score_prioridade_100",
    "score_valor_atual_100",
    "score_potencial_100",
    "score_crescimento_100",
    "score_risco_100",
    "motivo_principal",
    "acao_recomendada",
]

# Mantém todas as colunas originais e adiciona os resultados do modelo.
colunas_originais_disponiveis = [
    coluna for coluna in df_original.columns
    if coluna in df_segmentado.columns
]

colunas_exportacao = list(dict.fromkeys(
    colunas_originais_disponiveis + colunas_modelo
))

df_exportacao = df_segmentado[colunas_exportacao].copy()

try:
    df_exportacao.to_parquet(
        "clientes_segmentados.parquet",
        index=False
    )
    print("Arquivo criado: clientes_segmentados.parquet")
except Exception as erro:
    print(f"Não foi possível gerar Parquet: {erro}")
    df_exportacao.to_csv(
        "clientes_segmentados.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("Arquivo criado: clientes_segmentados.csv")

if SALVAR_CSV_COMPLETO:
    df_exportacao.to_csv(
        "clientes_segmentados.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print("Arquivo criado: clientes_segmentados.csv")

resumo_perfis.to_csv(
    "resumo_segmentacao.csv",
    index=False,
    encoding="utf-8-sig"
)

centroides_perfis.to_csv(
    "centroides_segmentacao.csv",
    index=False,
    encoding="utf-8-sig"
)

df_diagnostico.to_csv(
    "diagnostico_segmentacao.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos auxiliares criados:")
print("- resumo_segmentacao.csv")
print("- centroides_segmentacao.csv")
print("- diagnostico_segmentacao.csv")



## 12. Sugestão de página no Power BI

### Visuais principais

- **Cards:** clientes, ROB total, ROB médio, produtos por cliente e percentual em risco.
- **Matriz:** segmento interno nas linhas e arquétipo nas colunas.
- **Dispersão:** `score_valor_atual_100` no eixo X e `score_potencial_100` no eixo Y.
- **Tamanho da bolha:** faturamento ou ROB.
- **Tabela operacional:** CNPJ, perfil, prioridade, motivo e ação recomendada.
- **Distribuição:** percentual dos quatro perfis por Diretoria, Regional, Agência e Gerente.
- **Drill-through do cliente:** histórico de ROB, produtos, engajamento, rating e restrição.

### Ordem recomendada de atuação

1. **Em Risco / Prioridade A:** preservar valor relevante antes da deterioração.
2. **Foco em Conversão / Prioridade A:** converter alto potencial em relacionamento.
3. **Crescimento em Valor / Prioridade A:** acelerar clientes com tração recente.
4. **Top Performance / Prioridade A:** proteger e aprofundar os melhores clientes.

### Próxima evolução

Para produção mensal, salve uma fotografia por `DT_MESREF`. Isso permitirá medir migrações entre os quatro perfis, taxa de conversão, ROB incremental e efetividade das ações.
